<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 99.9M/99.9M [00:01<00:00, 90.3MB/s]


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [5]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [7]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [8]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_1691/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-22  114.983864  174.864944  133.651230  127.080002  98.782745   
2023-08-23  116.440216  178.703003  132.605392  128.130005  98.763969   
2023-08-24  116.792091  174.026276  132.199661  124.720001  97.467537   
2023-08-25  116.977814  176.226517  132.253784  125.790001  97.899666   
2023-08-28  117.212379  177.785446  132.920944  126.154999  96.565659   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-22  70.908501  292.468719  519.479980  168.299362  73.060760  ...   
2023-08-23  71.697754  298.566162  530.710022  169.080521  73.386406  ...   
2023-08-24  71.707260  297.430054  512.429993  165.469879  72.662766  ...   
2023-08-25  70.956047  301.804230  525.059998  169.728348  73.115044  ...   
2023-08-28  70.547165  303.839905  529.919983  171.795639  73.151215  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2023-08-22  4728200  2296400  3142600  10518800   833700  10494000  1207700   
2023-08-23  3083400  1298600  8610700  11435600  1012500  12207300  1441600   
2023-08-24  1858400  1494000  5780900  10818500   896800   7952700  1131100   
2023-08-25  2768500  1706200  7705600  13579300   973800   7741000  1097300   
2023-08-28  1360500  1209800  3982200  10382100   803600   5333600  1815600   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-22  3648900  340500  1286400  
2023-08-23  2755200  314800  1022900  
2023-08-24  1953000  246300  1210300  
2023-08-25  1456700  337300   879700  
2023-08-28  2658500  211000  1546900  

[5 rows x 2515 columns]

In [9]:
closes = df["Close"]
returns = closes.pct_change() *100

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
fdf = pd.concat([cs,rs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_1691/2048977385.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return
495,2023-08-23,A,116.440216,1.266571
496,2023-08-23,AAPL,178.703003,2.194870
497,2023-08-23,ABBV,132.605392,-0.782512
498,2023-08-23,ABNB,128.130005,0.826254
499,2023-08-23,ABT,98.763969,-0.019007
500,2023-08-23,ACGL,71.697754,1.113059
501,2023-08-23,ACN,298.566162,2.084819
502,2023-08-23,ADBE,530.710022,2.161785
503,2023-08-23,ADI,169.080521,0.464148
504,2023-08-23,ADM,73.386406,0.445719


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [10]:
fdf_appl = fdf[fdf['ticker'] == 'AAPL'].copy()
fdf_appl.head()

,date,ticker,close,return
496,2023-08-23,AAPL,178.703003,2.194870
991,2023-08-24,AAPL,174.026276,-2.617039
1486,2023-08-25,AAPL,176.226517,1.264315
1981,2023-08-28,AAPL,177.785446,0.884617
2476,2023-08-29,AAPL,181.662994,2.181027


In [11]:
len(fdf_appl)

752

In [12]:
fdf_appl['Lag 1'] = fdf_appl['close'].shift(1)
fdf_appl['Lag 2'] = fdf_appl['close'].shift(2)
fdf_appl['Lag 3'] = fdf_appl['close'].shift(3)
fdf_appl['Lag 5'] = fdf_appl['close'].shift(5)
fdf_appl.head(25)

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5
496,2023-08-23,AAPL,178.703003,2.194870,NaN,NaN,NaN,NaN
991,2023-08-24,AAPL,174.026276,-2.617039,178.703003,NaN,NaN,NaN
1486,2023-08-25,AAPL,176.226517,1.264315,174.026276,178.703003,NaN,NaN
1981,2023-08-28,AAPL,177.785446,0.884617,176.226517,174.026276,178.703003,NaN
2476,2023-08-29,AAPL,181.662994,2.181027,177.785446,176.226517,174.026276,NaN
2971,2023-08-30,AAPL,185.145889,1.917229,181.662994,177.785446,176.226517,178.703003
3466,2023-08-31,AAPL,185.362930,0.117227,185.145889,181.662994,177.785446,174.026276
3961,2023-09-01,AAPL,186.931717,0.846332,185.362930,185.145889,181.662994,176.226517
4456,2023-09-05,AAPL,187.168518,0.126678,186.931717,185.362930,185.145889,177.785446
4951,2023-09-06,AAPL,180.469162,-3.579318,187.168518,186.931717,185.362930,181.662994


In [13]:
fdf_appl['SMA 5'] = fdf_appl['close'].rolling(window=5).mean()
fdf_appl['SMA 10'] = fdf_appl['close'].rolling(window=10).mean()
fdf_appl['SMA 20'] = fdf_appl['close'].rolling(window=20).mean()
fdf_appl['SMA 30'] = fdf_appl['close'].rolling(window=30).mean()
fdf_appl['SMA 50'] = fdf_appl['close'].rolling(window=50).mean()

fdf_appl.head(100)
#fdf_appl = fdf[fdf['ticker'] == 'AAPL'].copy()
#fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
496,2023-08-23,AAPL,178.703003,2.194870,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
991,2023-08-24,AAPL,174.026276,-2.617039,178.703003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,2023-08-25,AAPL,176.226517,1.264315,174.026276,178.703003,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1981,2023-08-28,AAPL,177.785446,0.884617,176.226517,174.026276,178.703003,NaN,NaN,NaN,NaN,NaN,NaN
2476,2023-08-29,AAPL,181.662994,2.181027,177.785446,176.226517,174.026276,NaN,177.680847,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
47587,2024-01-09,AAPL,182.910034,-0.226345,183.324982,178.997742,179.718948,183.404007,181.396490,185.339415,189.364345,189.337009,185.741103
48083,2024-01-10,AAPL,183.947403,0.567147,182.910034,183.324982,178.997742,182.030746,181.779822,184.661679,189.019056,189.218456,186.100547
48579,2024-01-11,AAPL,183.354614,-0.322260,183.947403,182.910034,183.324982,179.718948,182.506955,183.914786,188.568549,189.060053,186.407289
49075,2024-01-12,AAPL,183.680634,0.177808,183.354614,183.947403,182.910034,178.997742,183.443533,183.158012,187.973798,188.946439,186.711078


In [14]:
fdf_appl['volatility 5'] = fdf_appl['return'].rolling(5).std()
fdf_appl['volatility 10'] = fdf_appl['return'].rolling(10).std()
fdf_appl['volatility 20'] = fdf_appl['return'].rolling(20).std()
fdf_appl['volatility 30'] = fdf_appl['return'].rolling(30).std()
fdf_appl.head()

,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
496,2023-08-23,AAPL,178.703003,2.194870,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
991,2023-08-24,AAPL,174.026276,-2.617039,178.703003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,2023-08-25,AAPL,176.226517,1.264315,174.026276,178.703003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1981,2023-08-28,AAPL,177.785446,0.884617,176.226517,174.026276,178.703003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2476,2023-08-29,AAPL,181.662994,2.181027,177.785446,176.226517,174.026276,NaN,177.680847,NaN,NaN,NaN,NaN,1.98432,NaN,NaN,NaN


In [15]:
fdf_appl = fdf_appl.dropna()
#fdf_appl = fdf_appl.drop(columns = 'SMA 100')
fdf_appl.head()


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
24771,2023-11-01,AAPL,171.648422,1.873841,168.491165,168.017532,165.975174,168.816757,167.759042,169.312044,172.510287,171.867807,174.296031,1.673387,1.324765,1.092395,1.082174
25267,2023-11-02,AAPL,175.200409,2.069338,171.648422,168.491165,168.017532,164.662918,169.866541,169.520227,172.641512,171.987521,174.225979,0.742187,1.489234,1.176677,1.134112
25763,2023-11-03,AAPL,174.292633,-0.518136,175.200409,171.648422,168.491165,165.975174,171.530032,169.892189,172.600070,172.048693,174.231306,1.093144,1.403534,1.136200,1.136273
26259,2023-11-06,AAPL,176.838242,1.460537,174.292633,175.200409,171.648422,168.017532,173.294174,170.506874,172.611909,172.152292,174.243540,1.110628,1.454435,1.168664,1.158974
26755,2023-11-07,AAPL,179.393692,1.445078,176.838242,174.292633,175.200409,168.491165,175.474680,171.333694,172.781122,172.476573,174.275705,1.032942,1.492702,1.207872,1.092175


In [16]:
len(fdf_appl)

703

In [17]:
fdf_appl['future return'] = fdf_appl.groupby('ticker')['return'].shift(-1)
fdf_appl = fdf_appl.dropna()
print(len(fdf_appl))
fdf_appl.head(len(fdf_appl))


702


,date,ticker,close,return,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
24771,2023-11-01,AAPL,171.648422,1.873841,168.491165,168.017532,165.975174,168.816757,167.759042,169.312044,172.510287,171.867807,174.296031,1.673387,1.324765,1.092395,1.082174,2.069338
25267,2023-11-02,AAPL,175.200409,2.069338,171.648422,168.491165,168.017532,164.662918,169.866541,169.520227,172.641512,171.987521,174.225979,0.742187,1.489234,1.176677,1.134112,-0.518136
25763,2023-11-03,AAPL,174.292633,-0.518136,175.200409,171.648422,168.491165,165.975174,171.530032,169.892189,172.600070,172.048693,174.231306,1.093144,1.403534,1.136200,1.136273,1.460537
26259,2023-11-06,AAPL,176.838242,1.460537,174.292633,175.200409,171.648422,168.017532,173.294174,170.506874,172.611909,172.152292,174.243540,1.110628,1.454435,1.168664,1.158974,1.445078
26755,2023-11-07,AAPL,179.393692,1.445078,176.838242,174.292633,175.200409,168.491165,175.474680,171.333694,172.781122,172.476573,174.275705,1.032942,1.492702,1.207872,1.092175,0.588488
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
372950,2026-08-14,AAPL,305.929993,0.219479,305.260010,302.250000,304.910004,313.059998,305.322003,307.481482,318.219887,318.552157,308.950681,1.036325,1.194857,2.144831,1.989030,-0.111135
373453,2026-08-17,AAPL,305.589996,-0.111135,305.929993,305.260010,302.250000,308.260010,304.788000,307.724625,317.183958,318.325470,308.843245,0.843827,1.038062,2.106093,1.973547,1.452928
373956,2026-08-18,AAPL,310.029999,1.452928,305.589996,305.929993,305.260010,304.910004,305.812000,307.816284,316.312579,318.313393,308.902341,0.916060,0.943478,2.138330,1.989058,2.193332
374459,2026-08-19,AAPL,316.829987,2.193332,310.029999,305.589996,305.929993,302.250000,308.727997,308.426080,315.873619,318.437061,309.213337,0.930132,1.162220,2.204919,2.022921,-1.745415


In [18]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50']
x = fdf_appl[features]
y = fdf_appl['future return']



In [19]:
split_index = int(0.8 * len(fdf_appl))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [20]:
from fastai.tabular.all import *
split_index = int(len(fdf_appl)*.8)


In [24]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error as mse

model = LinearRegression()
model.fit(x_train, y_train)
predictions = model.predict(x_test)

loss = mse(y_test, predictions)
loss






4.097722513310679